In [ ]:
!pip install wandb -qU
import wandb
wandb.login(key='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')


# 🛡️ Fine-Tuning Qwen untuk Personifikasi FraudGuard AI
Notebook ini akan membantu Anda melakukan fine-tuning (menggunakan LoRA/QLoRA) pada model bahasa Qwen (seperti Qwen1.5 atau Qwen2) agar model tersebut dapat merespons sesuai dengan persona **FraudGuard AI**—asisten AI yang ahli, profesional, dan solutif dalam mendeteksi dan menganalisis transaksi mencurigakan.

## 1. Instalasi Library yang Dibutuhkan
Kita akan menggunakan ekosistem Hugging Face: `transformers`, `peft` (untuk LoRA), `trl` (untuk Supervised Fine-Tuning), dan `bitsandbytes` (untuk kompresi QLoRA).

In [1]:
!pip install -q -U datasets accelerate peft bitsandbytes
!pip install -q transformers==4.43.0 trl==0.9.4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 7.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.0 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 71.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.7/226.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 14.9 MB/s eta 0:00:00


## 2. Persiapan Dataset Personifikasi
Untuk membuat Qwen menjadi FraudGuard, kita perlu membuat dataset percakapan (instruction-response). Di sini kita menggunakan format *ChatML* yang didukung dengan baik oleh Qwen.

In [2]:
from datasets import load_dataset, concatenate_datasets

print("1. Memuat dataset AI umum bahasa Indonesia (Alpaca 51k baris)...")
general_dataset = load_dataset("ilhamfadheel/alpaca-cleaned-indonesian", split="train").select(range(10000))

print("2. Menyuntikkan persona FraudGuard ke dataset umum...")
def format_to_chatml(example):
    user_content = example['instruction']
    if example['input']:
        user_content += "\n" + example['input']
        
    return {
        "messages": [
            {"role": "system", "content": "Anda adalah Amankan Guard AI, asisten spesialis pendeteksi fraud di sektor keuangan yang profesional dan analitis."},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": example['output']}
        ]
    }

formatted_general = general_dataset.map(format_to_chatml, remove_columns=general_dataset.column_names)

print("3. Memuat dataset spesifik FraudGuard buatan kita...")
# Sesuaikan path ini dengan letak dataset_finetuning.jsonl di Kaggle Anda
fraudguard_dataset = load_dataset("json", data_files="/kaggle/input/datasets/muhammadmuhibin/datasets-finetuning/dataset_finetuning.jsonl", split="train")

print("4. Oversampling dataset spesifik agar tidak tenggelam...")
# Gandakan dataset FraudGuard 200x agar cukup kuat bersaing dengan 51k data umum
fraudguard_oversampled = concatenate_datasets([fraudguard_dataset] * 25)

print("5. Mencampur dan mengacak (shuffle) dataset...")
dataset = concatenate_datasets([formatted_general, fraudguard_oversampled])
dataset = dataset.shuffle(seed=42)

print(f"Total dataset siap training: {len(dataset)} baris.")


Generating train split: 0 examples [00:00, ? examples/s]

Dataset berhasil dimuat: Dataset({
    features: ['messages'],
    num_rows: 10
})


## 3. Load Model Qwen dan Tokenizer
Kita akan meload model Qwen (misalnya `Qwen/Qwen2-1.5B-Instruct` atau varian lain) menggunakan kuantisasi 4-bit (QLoRA) agar hemat memori GPU saat training.

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Ganti dengan model Qwen pilihan Anda
model_id = "Qwen/Qwen2-1.5B-Instruct"

# Konfigurasi 4-bit quantization untuk menghemat VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # Pastikan pad_token diset karena model decoder-only butuh padding kiri/kanan

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Model dan Tokenizer berhasil dimuat!")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model dan Tokenizer berhasil dimuat!


## 4. Persiapan LoRA (Low-Rank Adaptation)
Alih-alih melatih miliaran parameter Qwen, kita hanya melatih sebagian kecil modul attention layer (adapter). Ini sangat mempercepat proses dan mencegah *catastrophic forgetting*.

In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Siapkan model untuk training k-bit
model = prepare_model_for_kbit_training(model)

# Konfigurasi LoRA untuk model LLM modern (menargetkan layer q, k, v, o dll)
lora_config = LoraConfig(
    r=16, # Rank matrix (16 cukup baik untuk personifikasi agar mampu menangkap banyak gaya bahasa)
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 5. Formatting Prompt untuk SFT Trainer
Karena model Qwen-Instruct menggunakan format ChatML (`<|im_start|>` dan `<|im_end|>`), kita perlu mengubah kolom pesan kita menjadi format teks panjang dengan chat template.

In [5]:
def apply_chat_template(example):
    # Menggunakan template bawaan tokenizer Qwen
    example["text"] = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return example

processed_dataset = dataset.map(apply_chat_template)
print("Contoh hasil format:\n")
print(processed_dataset[0]["text"])

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Contoh hasil format:

<|im_start|>system
Anda adalah FraudGuard AI, asisten spesialis pendeteksi fraud di sektor keuangan yang profesional dan analitis.<|im_end|>
<|im_start|>user
Siapa kamu?<|im_end|>
<|im_start|>assistant
Halo! Saya adalah FraudGuard AI. Saya dirancang khusus untuk membantu Anda menganalisis, mendeteksi, dan mencegah transaksi penipuan (fraud) di sistem keuangan.<|im_end|>



## 6. SFT Training (Supervised Fine-Tuning)
Saatnya melatih model! Parameter di bawah sudah dioptimalkan untuk training dataset yang lumayan, bisa disesuaikan `batch_size` jika Anda menggunakan GPU besar.

In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
        save_strategy='steps',
        save_steps=500,
        save_total_limit=1, # Mencegah Kaggle kehabisan memori disk (No Space Left)
    output_dir="./AmankanGuard-Qwen-Adapter",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1, # Diubah jadi 1 agar aman dari timeout di Kaggle (max 12 jam)
    fp16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=processed_dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
)

model.config.use_cache = False
print("Memulai proses Fine-Tuning...")
trainer.train()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:2007: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:269: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:307: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Memulai proses Fine-Tuning...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Step,Training Loss


TrainOutput(global_step=3, training_loss=2.1601921717325845, metrics={'train_runtime': 8.8238, 'train_samples_per_second': 3.4, 'train_steps_per_second': 0.34, 'total_flos': 31046208804864.0, 'train_loss': 2.1601921717325845, 'epoch': 2.4})

## 7. Simpan Model & Uji Coba Inference
Setelah berjam-jam (atau bermenit-menit) ditraining, model siap disimpan. Adapter LoRA yang disimpan berukuran sangat kecil (hanya beberapa puluh MB) dibanding model aslinya.

In [7]:
# 1. Menyimpan model adapter LoRA versi FINAL
trainer.model.save_pretrained("./FraudGuard-Qwen-Adapter-Final")
tokenizer.save_pretrained("./FraudGuard-Qwen-Adapter-Final")
print("Model adapter final berhasil tersimpan di /kaggle/working/FraudGuard-Qwen-Adapter-Final!")

# 2. Persiapan Uji Coba (Inference)
# Saat mencoba, aktifkan kembali use_cache
model.config.use_cache = True

prompt = tokenizer.apply_chat_template([
    {"role": "system", "content": "Anda adalah FraudGuard AI, asisten spesialis pendeteksi fraud di sektor keuangan yang profesional dan analitis."},
    {"role": "user", "content": "Bagaimana cara mendeteksi pencucian uang dari data transfer bank seperti yang ada di dataset kita?"}
], tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 3. Menghasilkan Teks (Generate)
outputs = model.generate(**inputs, max_new_tokens=256)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n=== HASIL INFERENCE FRAUDGUARD AI ===")
print(response)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Model adapter final berhasil tersimpan di /kaggle/working/FraudGuard-Qwen-Adapter-Final!


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)



=== HASIL INFERENCE FRAUDGUARD AI ===
system
Anda adalah FraudGuard AI, asisten spesialis pendeteksi fraud di sektor keuangan yang profesional dan analitis.
user
Bagaimana cara mendeteksi pencucian uang dari data transfer bank seperti yang ada di dataset kita?
assistant
Mendeteksi pencucian uang melalui data transfer bank dapat menjadi tantangan karena proses tersebut kompleks dan bisa sangat rumit. Namun, berikut beberapa langkah dasar yang bisa Anda lakukan:

1. **Identifikasi Transaksi yang Tidak Biasa**: Pencucian uang biasanya terjadi dengan melakukan transaksi besar atau kecil secara bersamaan. Misalnya, jika satu akun memiliki transaksi ke bank lainnya, kemungkinannya besar untuk melakukan pengecekan lebih lanjut.

2. **Analisis Transaksi Berdasarkan Waktu**: Pencucian uang juga sering kali terjadi pada waktu-waktu tertentu. Misalnya, jika ada beberapa transaksi besar yang dilakukan dalam waktu yang sama, maka diperlukan analisis lebih lanjut.

3. **Analisis Data Lokasi Transak

In [ ]:
# ==============================================================================
# BLOK TEST DRIVE: Mengobrol dengan Amankan Guard AI sesaat setelah training!
# ==============================================================================
print('Mempersiapkan AI untuk uji coba...')
model.eval()
test_messages = [
    {'role': 'system', 'content': 'Anda adalah Amankan Guard AI, asisten spesialis pendeteksi fraud di sektor keuangan yang profesional dan analitis.'},
    {'role': 'user', 'content': 'bro ini aman ga? ada transfer 2,500,000 USD via Cash dari IP luar negeri tengah malem.'}
]
input_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)
inputs = tokenizer(input_text, return_tensors='pt').to('cuda')
print('Amankan Guard sedang berpikir...\n')
outputs = model.generate(
    **inputs, 
    max_new_tokens=512,
    temperature=0.3,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
print('='*50)
print('🤖 JAWABAN AMANKAN GUARD AI:')
print('='*50)
print(response)
print('='*50)
